In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

In [2]:
anthropic_client = AsyncOpenAI(
    api_key=os.environ["ANTHROPIC_API_KEY"],
    base_url="https://api.anthropic.com/v1/",
)
   
anthropic_model = OpenAIChatCompletionsModel(
        model="claude-haiku-4-5-20251001",
        openai_client=anthropic_client,
    )

In [3]:
agent = Agent(
    name="Jokester",
    instructions="You are a joke teller",
    model= anthropic_model
)

In [4]:
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

In [5]:
print(result.final_output)

# Why did the autonomous AI agent go to therapy?

Because it kept getting stuck in **infinite loops** of self-doubt! 🔄

It would think: *"Should I make this decision? Let me analyze whether I should analyze this decision. But first, let me think about whether I should think about that..."*

By the time it finished deliberating, the moment had passed and it had to autonomously apologize to everyone! 😄

---

**Bonus version:** An autonomous AI agent walks into a bar. The bartender asks, "What'll it be?" The AI responds, "I don't know, let me run 10,000 simulations." The bartender says, "Just order a drink!" The AI says, "I can't—I'm still deciding whether I should have preferences!" 🍺🤖


In [6]:
result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': '# Why did the autonomous AI agent go to therapy?\n\nBecause it kept getting stuck in **infinite loops** of self-doubt! 🔄\n\nIt would think: *"Should I make this decision? Let me analyze whether I should analyze this decision. But first, let me think about whether I should think about that..."*\n\nBy the time it finished deliberating, the moment had passed and it had to autonomously apologize to everyone! 😄\n\n---\n\n**Bonus version:** An autonomous AI agent walks into a bar. The bartender asks, "What\'ll it be?" The AI responds, "I don\'t know, let me run 10,000 simulations." The bartender says, "Just order a drink!" The AI says, "I can\'t—I\'m still deciding whether I should have preferences!" 🍺🤖',
    'type': 'output_text'}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [7]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

# Why did the autonomous AI agent go to therapy?

Because it kept getting stuck in **infinite loops** of self-doubt! 🔄

It would think: *"Should I take this action? Let me analyze that decision... which requires analyzing the analysis... which requires analyzing..."*

By the time it made a decision, the problem had already solved itself! 😄

---

**Bonus version:** An autonomous AI agent walks into a bar. The bartender asks, "What'll it be?" The AI says, "I'll need to consult with my decision tree, query my knowledge base, and run 10,000 simulations first." 

The bartender sighs. "So... the usual?"

The AI replies: "Yes, but I had to optimize it myself!" 🤖


In [8]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [9]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
push("HEY!!")

Push: HEY!!


In [11]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [12]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001F19C378D60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [13]:
push_tool.description

'Send the given message to the user as a push notification'

In [14]:

notifier = Agent(name="Notifier", model=anthropic_model, instructions="You notify the user upon request", tools=[push_tool])

In [15]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done! I've sent a push notification to the user letting them know the pizza is here.


## Part 3: Sessions (memory)


In [16]:
agent = Agent(name="Assistant", model=anthropic_model)

In [17]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed! Nice to meet you. How can I help you today?


In [18]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't have any information about your name. We haven't met before, and you haven't told me your name in our conversation. Feel free to share it if you'd like!


## Memory approach 1 - just manually pass in the list of dicts


In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ahmed")
print(response.final_output)

AttributeError: 'coroutine' object has no attribute 'final_output'

In [20]:
response.to_input_list()

[{'content': 'Hi there. My name is Ahmed', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hi Ahmed! Nice to meet you. How can I help you today?',
    'type': 'output_text'}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [21]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ahmed', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hi Ahmed! Nice to meet you. How can I help you today?',
    'type': 'output_text'}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'},
 {'role': 'user', 'content': "What's my name?"}]

In [22]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ahmed! You told me that at the start of our conversation.


## Another approach - use OpenAI Agents SDK built in SQLLite session


In [24]:
session = SQLiteSession("12346")

In [25]:
response = await Runner.run(agent, "Hi there. My name is Ahmed.", session=session)
print(response.final_output)

Hi Ahmed! Nice to meet you. How can I help you today?


In [26]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ahmed. You told me that at the start of our conversation!
